In [1]:
from __future__ import annotations

from pathlib import Path

import io
import zipfile
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests  # used only if we download Fama-French data
except Exception:
    requests = None

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

In [2]:
if importlib.util.find_spec('statsmodels') is None:
    raise ModuleNotFoundError(
        "Lecture 4 requires statsmodels. In Jupyter, run: %pip install statsmodels"
    )

import statsmodels.api as sm

In [3]:
import re

DATA_PATH = 'Charting Excel Export - Stocks.xls'

START_DATE = pd.Timestamp('01/01/1998')
END_DATE   = pd.Timestamp('31/12/2024')


def _load_charting_xls(path: str) -> pd.DataFrame:
    """
    Read an Excel Export file (Pane 1 sheet) from Capital IQ.
    Returns a beginning-of-month DatetimeIndex DataFrame.

    Capital IQ labels end-of-month prices on the 1st of the following month
    when the month ends on a weekend or holiday. Shift those dates back one day
    so they are grouped with the correct prior month before resampling.
    """
    raw = pd.read_excel(path, sheet_name='Pane 1', header=None)
    mask = raw.notna().any(axis=1)
    data = raw[mask].copy()
    data.columns = data.iloc[0]          # first surviving row is the header
    data = data.iloc[1:].copy()
    data = data.rename(columns={data.columns[0]: 'date'})
    data['date'] = pd.to_datetime(data['date'])
    data = data.set_index('date').apply(pd.to_numeric, errors='coerce')
    # Dates on the 1st of a month are end-of-prior-month prices reported on the
    # next business day. Shift them back one day into the correct month.
    data.index = data.index - pd.to_timedelta((data.index.day == 1).astype(int), unit='D')
    # Group by calendar month (last observation), then label beginning-of-month
    # to match the Fama-French date convention.
    data = data.resample('ME').last()
    data.index = data.index.to_period('M').to_timestamp()
    return data


def _extract_ticker(col: str) -> str:
    """Pull the ticker symbol out of a long Charting column label."""
    m = re.search(r'\((?:[A-Za-z]+:)?([A-Z]+)\)', col)
    return m.group(1) if m else col


# ── Load all assets from a single file ────────────────────────────────────────
all_prices = _load_charting_xls(DATA_PATH)
all_prices.columns = [_extract_ticker(c) for c in all_prices.columns]

# Dividend-adjusted price levels → monthly simple return
# r_t = P_t / P_{t-1} - 1
all_ret = all_prices.pct_change().dropna(how='all')

# ── Split stocks and S&P 500 proxy (SPY) ──────────────────────────────────────
stock_tickers = [c for c in all_ret.columns if c != 'SPY']
stocks_prices = all_prices[stock_tickers]   # kept for stock-selection cell below
stocks_ret    = all_ret[stock_tickers]
spy_ret       = all_ret[['SPY']]

# ── Merge stocks + SPY ────────────────────────────────────────────────────────
df_ret = stocks_ret.join(spy_ret, how='inner').dropna(how='all')
df_ret.index.name = 'date'

df_ret = df_ret.loc[START_DATE:END_DATE]

print(f'Data file    : {DATA_PATH}')
print(f'Rows (months): {df_ret.shape[0]}, Columns (assets): {df_ret.shape[1]}')
print(f'Date range   : {df_ret.index.min().date()} to {df_ret.index.max().date()}')
df_ret.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Charting Excel Export - Stocks.xls'